# FLAN-T5-small Full Fine-Tuning: fixed hyperparameter search


In [ ]:
%cd /content
!rm -rf LORA-course-project
!git clone https://github.com/NikitaBaranenkov/LORA-course-project.git
%cd LORA-course-project
!ls
!ls src

/content
Cloning into 'LORA-course-project'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 105 (delta 27), reused 80 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 753.60 KiB | 5.27 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/LORA-course-project
artifacts  data    notebooks  reports		src
configs    legacy  README.md  requirements.txt
constants.py  __init__.py  README.md


In [ ]:
%pip uninstall -y torchao
%pip install -q --upgrade transformers accelerate datasets evaluate scikit-learn sentencepiece

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.1 MB/s eta 0:00:00


## Google Drive setup for artifact backup

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ARTIFACTS_ROOT = Path('/content/drive/MyDrive/lora_course_project/artifacts')
DRIVE_ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

test_file = DRIVE_ARTIFACTS_ROOT / 'drive_write_test.txt'
test_file.write_text('Google Drive write test passed', encoding='utf-8')

print(f'Google Drive is mounted and writable: {DRIVE_ARTIFACTS_ROOT}')

Mounted at /content/drive
Google Drive is mounted and writable: /content/drive/MyDrive/lora_course_project/artifacts


## Imports and experiment paths

In [ ]:
import gc
import inspect
import json
import random
import time
from itertools import product

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

from src.constants import (
    DATASET_NAME,
    FINAL_TEST_EVALUATION_ONLY,
    MAX_LENGTH,
    PROJECT_ROOT,
    REPORTS_TABLES_DIR,
    SEED,
    SELECTION_METRIC,
    TEST_CSV_PATH,
    TRAIN_CSV_PATH,
    VALIDATION_CSV_PATH,
    id2label,
    label2id,
)

MODEL_ID = 'google/flan-t5-small'
RUN_NAME = 'flan_t5_small_full_ft_hparam_search'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts' / RUN_NAME
CHECKPOINTS_DIR = ARTIFACTS_DIR / 'checkpoints'
BEST_MODEL_DIR = ARTIFACTS_DIR / 'best_model'
RUN_TABLES_DIR = REPORTS_TABLES_DIR / 'flan_t5_small_full_ft'

SWEEP_RESULTS_PATH = RUN_TABLES_DIR / 'flan_t5_small_full_ft_hparam_search_validation_results.csv'
FINAL_TEST_RESULTS_PATH = RUN_TABLES_DIR / 'flan_t5_small_full_ft_best_hparam_test_results.csv'
EXPERIMENT_CONFIG_PATH = RUN_TABLES_DIR / 'flan_t5_small_full_ft_hparam_search_config.json'
GENERATION_DIAGNOSTICS_PATH = RUN_TABLES_DIR / 'flan_t5_small_full_ft_best_hparam_generation_diagnostics.csv'

MAX_SOURCE_LENGTH = MAX_LENGTH
MAX_TARGET_LENGTH = 8
COMMON_TRAINING_CONFIG = {
    'num_train_epochs': 3,
    'per_device_train_batch_size': 8,
    'per_device_eval_batch_size': 16,
    'gradient_accumulation_steps': 2,
    'warmup_ratio': 0.06,
    'fp16': False,
    'predict_with_generate': True,
    'generation_num_beams': 1,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'macro_f1',
    'greater_is_better': True,
    'save_total_limit': 1,
}

GENERATION_NUM_BEAMS = COMMON_TRAINING_CONFIG['generation_num_beams']
USE_GRADIENT_CHECKPOINTING = False
LOGGING_STEPS = 50

LABEL_IDS = sorted(id2label)
CLASS_NAMES = [id2label[class_id] for class_id in LABEL_IDS]
INVALID_LABEL_ID = -1

## Fixed data splits

In [ ]:
train_df = pd.read_csv(TRAIN_CSV_PATH).reset_index(drop=True)
validation_df = pd.read_csv(VALIDATION_CSV_PATH).reset_index(drop=True)
test_df = pd.read_csv(TEST_CSV_PATH).reset_index(drop=True)

for split_df in (train_df, validation_df, test_df):
    split_df['label'] = split_df['label'].astype(int)

raw_dataframes = {
    'train': train_df,
    'validation': validation_df,
    'test': test_df,
}

raw_datasets = DatasetDict({
    split_name: Dataset.from_pandas(split_df, preserve_index=False)
    for split_name, split_df in raw_dataframes.items()
})

print({split_name: len(split_df) for split_name, split_df in raw_dataframes.items()})
raw_datasets

{'train': 8105, 'validation': 1431, 'test': 2388}


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_name'],
        num_rows: 8105
    })
    validation: Dataset({
        features: ['text', 'label', 'label_name'],
        num_rows: 1431
    })
    test: Dataset({
        features: ['text', 'label', 'label_name'],
        num_rows: 2388
    })
})

## Text-to-text preprocessing

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

PROMPT_TEMPLATE = (
    'Classify the financial sentiment of the text. '
    'Return exactly one label: Bearish, Bullish, or Neutral. '
    'Text: {text} Sentiment:'
)


def format_source(text):
    return PROMPT_TEMPLATE.format(text=str(text))


def preprocess_batch(batch):
    sources = [format_source(text) for text in batch['text']]
    targets = [id2label[int(label)] for label in batch['label']]

    model_inputs = tokenizer(
        sources,
        padding='max_length',
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
    )
    target_tokens = tokenizer(
        text_target=targets,
        padding='max_length',
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
    )
    labels = [
        [token_id if token_id != tokenizer.pad_token_id else -100 for token_id in target_ids]
        for target_ids in target_tokens['input_ids']
    ]
    model_inputs['labels'] = labels
    return model_inputs


tokenized_datasets = raw_datasets.map(
    preprocess_batch,
    batched=True,
    remove_columns=raw_datasets['train'].column_names,
)
tokenized_datasets.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels'],
)

tokenized_train = tokenized_datasets['train']
tokenized_validation = tokenized_datasets['validation']
tokenized_test = tokenized_datasets['test']

sample = tokenized_train[0]
print('input_ids shape:', sample['input_ids'].shape)
print('attention_mask shape:', sample['attention_mask'].shape)
print('labels shape:', sample['labels'].shape)
print('source example:', format_source(train_df.loc[0, 'text']))
print('target example:', id2label[int(train_df.loc[0, 'label'])])
print('decoded target tokens:', tokenizer.decode(
    [token_id if token_id != -100 else tokenizer.pad_token_id for token_id in sample['labels'].tolist()],
    skip_special_tokens=True,
))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Map:   0%|          | 0/8105 [00:00<?, ? examples/s]

Map:   0%|          | 0/1431 [00:00<?, ? examples/s]

Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

input_ids shape: torch.Size([128])
attention_mask shape: torch.Size([128])
labels shape: torch.Size([8])
source example: Classify the financial sentiment of the text. Return exactly one label: Bearish, Bullish, or Neutral. Text: Oil's Biggest Bullish Boost Since 2016 Scores on Trade Armistice Sentiment:
target example: Bullish
decoded target tokens: Bullish


## Generation-based metrics

In [ ]:
LABEL_TEXT_TO_ID = {
    ''.join(label.lower().split()): class_id
    for class_id, label in id2label.items()
}
EXPECTED_METRIC_KEYS = {
    'accuracy',
    'macro_f1',
    'weighted_f1',
    'invalid_count',
    'invalid_rate',
}


def normalize_generated_label(text):
    compact = ''.join(str(text).strip().lower().split())
    if compact in LABEL_TEXT_TO_ID:
        return LABEL_TEXT_TO_ID[compact]

    hits = [label_text for label_text in LABEL_TEXT_TO_ID if label_text in compact]
    if len(hits) == 1:
        return LABEL_TEXT_TO_ID[hits[0]]
    return INVALID_LABEL_ID


def as_numpy_array(values):
    if isinstance(values, torch.Tensor):
        values = values.detach().cpu().numpy()
    return np.asarray(values)


def decode_token_ids(token_ids):
    token_ids = as_numpy_array(token_ids)
    if token_ids.ndim == 1:
        token_ids = token_ids.reshape(1, -1)
    token_ids = np.where(token_ids == -100, tokenizer.pad_token_id, token_ids)
    return tokenizer.batch_decode(token_ids, skip_special_tokens=True)


def main_metrics(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(
            y_true, y_pred, average='macro', labels=LABEL_IDS, zero_division=0
        )),
        'weighted_f1': float(f1_score(
            y_true, y_pred, average='weighted', labels=LABEL_IDS, zero_division=0
        )),
    }


def compute_seq2seq_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_predictions = decode_token_ids(predictions)
    decoded_labels = decode_token_ids(labels)

    y_pred = np.array([normalize_generated_label(text) for text in decoded_predictions])
    y_true = np.array([normalize_generated_label(text) for text in decoded_labels])

    metrics = main_metrics(y_true, y_pred)
    invalid_count = int(np.sum(y_pred == INVALID_LABEL_ID))
    invalid_rate = float(invalid_count / len(y_pred)) if len(y_pred) else 0.0
    return {
        'accuracy': metrics['accuracy'],
        'macro_f1': metrics['macro_f1'],
        'weighted_f1': metrics['weighted_f1'],
        'invalid_count': invalid_count,
        'invalid_rate': invalid_rate,
    }


metric_smoke_labels = tokenizer(
    text_target=['Bearish'],
    padding='max_length',
    truncation=True,
    max_length=MAX_TARGET_LENGTH,
)['input_ids']
metric_smoke = compute_seq2seq_metrics((
    np.array(metric_smoke_labels),
    np.array(metric_smoke_labels),
))
assert set(metric_smoke) == EXPECTED_METRIC_KEYS
assert SELECTION_METRIC == 'macro_f1'
assert SELECTION_METRIC in metric_smoke


## FLAN-T5 full fine-tuning model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def new_full_ft_model():
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
    if USE_GRADIENT_CHECKPOINTING:
        model.gradient_checkpointing_enable()
        model.config.use_cache = False
    return model


def summarize_parameters(model, print_summary=False):
    total_params = sum(parameter.numel() for parameter in model.parameters())
    frozen_names = [
        name for name, parameter in model.named_parameters() if not parameter.requires_grad
    ]
    trainable_params = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    trainable_share = trainable_params / total_params

    if print_summary:
        print(f'Total parameters:     {total_params:,}')
        print(f'Trainable parameters: {trainable_params:,}')
        print(f'Trainable share:      {100 * trainable_share:.4f}%')

    assert model.config.is_encoder_decoder
    assert not hasattr(model, 'classifier')
    assert not frozen_names, f'Full fine-tuning found frozen parameters: {frozen_names[:10]}'
    assert trainable_share > 0.9999, 'Full fine-tuning should train approximately 100% of parameters.'
    return {
        'total_params': total_params,
        'trainable_params': trainable_params,
        'trainable_share': trainable_share,
    }


set_global_seed(SEED)
preview_model = new_full_ft_model()
preview_parameter_summary = summarize_parameters(preview_model, print_summary=True)
del preview_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('Device:', device)
print('Base model:', MODEL_ID)
print('Adaptation: full fine-tuning')

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Total parameters:     76,961,152
Trainable parameters: 76,961,152
Trainable share:      100.0000%
Device: cuda
Base model: google/flan-t5-small
Adaptation: full fine-tuning


## Fixed grid for full fine-tuning

In [ ]:
FULL_FT_GRID = [
    {
        'learning_rate': learning_rate,
        'num_train_epochs': COMMON_TRAINING_CONFIG['num_train_epochs'],
        'weight_decay': weight_decay,
        'warmup_ratio': COMMON_TRAINING_CONFIG['warmup_ratio'],
        'per_device_train_batch_size': COMMON_TRAINING_CONFIG['per_device_train_batch_size'],
        'per_device_eval_batch_size': COMMON_TRAINING_CONFIG['per_device_eval_batch_size'],
        'gradient_accumulation_steps': COMMON_TRAINING_CONFIG['gradient_accumulation_steps'],
    }
    for learning_rate, weight_decay in product([1e-5, 2e-5, 3e-5], [0.0, 0.01])
]

print('Number of full fine-tuning configurations:', len(FULL_FT_GRID))
pd.DataFrame(FULL_FT_GRID)

Number of full fine-tuning configurations: 6


,learning_rate,num_train_epochs,weight_decay,warmup_ratio,per_device_train_batch_size,per_device_eval_batch_size,gradient_accumulation_steps
0,0.00001,3,0.00,0.06,8,16,2
1,0.00001,3,0.01,0.06,8,16,2
2,0.00002,3,0.00,0.06,8,16,2
3,0.00002,3,0.01,0.06,8,16,2
4,0.00003,3,0.00,0.06,8,16,2
5,0.00003,3,0.01,0.06,8,16,2


### Mandatory FLAN-T5 seq2seq diagnostics before the sweep

In [ ]:
assert SELECTION_METRIC == 'macro_f1'
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
RUN_TABLES_DIR.mkdir(parents=True, exist_ok=True)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

print('Label decoding after preprocessing:')
for i in range(10):
    labels = tokenized_train[i]['labels']
    labels = labels.tolist() if hasattr(labels, 'tolist') else labels
    real_label_tokens = [int(x) for x in labels if int(x) != -100]
    decoded_label = tokenizer.decode(real_label_tokens, skip_special_tokens=True)
    print(i, real_label_tokens, decoded_label)
    assert real_label_tokens, f'Example {i} has no non--100 target tokens.'
    assert decoded_label in CLASS_NAMES, f'Unexpected decoded label for example {i}: {decoded_label!r}'

batch = data_collator([tokenized_train[i] for i in range(4)])
print(batch.keys())
print(batch['labels'])
print((batch['labels'] != -100).sum(dim=1))
assert torch.all((batch['labels'] != -100).sum(dim=1) > 0)

set_global_seed(SEED)
model = new_full_ft_model().to(device)
summarize_parameters(model, print_summary=True)
model.eval()
model_device = getattr(model, 'device', next(model.parameters()).device)
batch = {k: v.to(model_device) for k, v in batch.items()}
with torch.no_grad():
    outputs = model(**batch)
print(outputs.loss)
assert torch.isfinite(outputs.loss).item(), f'One-batch loss is not finite: {outputs.loss}'
assert float(outputs.loss.detach().cpu()) > 0.0, f'One-batch loss should not be zero: {outputs.loss}'

def trainer_tokenizer_kwargs():
    signature = inspect.signature(Seq2SeqTrainer.__init__)
    key = 'processing_class' if 'processing_class' in signature.parameters else 'tokenizer'
    return {key: tokenizer}


diagnostic_eval_args_kwargs = {
    'output_dir': str(CHECKPOINTS_DIR / 'diagnostics'),
    'per_device_eval_batch_size': min(4, FULL_FT_GRID[0]['per_device_eval_batch_size']),
    'report_to': 'none',
    'fp16': COMMON_TRAINING_CONFIG['fp16'],
    'prediction_loss_only': False,
    'label_names': ['labels'],
    'predict_with_generate': COMMON_TRAINING_CONFIG['predict_with_generate'],
    'generation_max_length': MAX_TARGET_LENGTH,
    'generation_num_beams': COMMON_TRAINING_CONFIG['generation_num_beams'],
    'remove_unused_columns': False,
}
signature = inspect.signature(Seq2SeqTrainingArguments.__init__)
if 'eval_strategy' in signature.parameters:
    diagnostic_eval_args_kwargs['eval_strategy'] = 'no'
elif 'evaluation_strategy' in signature.parameters:
    diagnostic_eval_args_kwargs['evaluation_strategy'] = 'no'
if 'logging_nan_inf_filter' in signature.parameters:
    diagnostic_eval_args_kwargs['logging_nan_inf_filter'] = False

diagnostic_eval_dataset = tokenized_validation.select(range(min(8, len(tokenized_validation))))
trainer = Seq2SeqTrainer(
    model=model,
    args=Seq2SeqTrainingArguments(**diagnostic_eval_args_kwargs),
    eval_dataset=diagnostic_eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_seq2seq_metrics,
    **trainer_tokenizer_kwargs(),
)
eval_results = trainer.evaluate()
print(eval_results.keys())
print(eval_results)
assert 'eval_macro_f1' in eval_results
assert 'eval_loss' in eval_results
assert np.isfinite(eval_results['eval_loss']), eval_results

assert COMMON_TRAINING_CONFIG['fp16'] is False, (
    'Keep FLAN-T5 training in fp32 unless a separate stability check proves mixed precision is safe.'
)

del trainer, model, outputs, batch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Label decoding after preprocessing:
0 [10204, 1273, 1] Bullish
1 [3617, 8792, 1] Neutral
2 [10204, 1273, 1] Bullish
3 [3617, 8792, 1] Neutral
4 [10204, 1273, 1] Bullish
5 [3617, 8792, 1] Neutral
6 [9034, 1273, 1] Bearish
7 [3617, 8792, 1] Neutral
8 [3617, 8792, 1] Neutral
9 [3617, 8792, 1] Neutral
KeysView({'input_ids': tensor([[ 4501,  4921,     8,   981,  6493,    13,     8,  1499,     5,  9778,
          1776,    80,  3783,    10,  9034,  1273,     6, 10204,  1273,     6,
            42,  3617,  8792,     5,  5027,    10,  6067,    31,     7,  2734,
           122,   222, 10204,  1273,     3, 16481,  1541,  1421, 17763,     7,
            30,  6550,  5412,  3040,    15,  4892,  2998,   295,    10,     1,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,   

/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Total parameters:     76,961,152
Trainable parameters: 76,961,152
Trainable share:      100.0000%
tensor(0.6045, device='cuda:0')


Training Loss,Validation Loss,Step,Accuracy,Macro F1,Weighted F1,Invalid Count,Invalid Rate
No log,0.239257,0,0.875000,0.311111,0.816667,0,0.000000


dict_keys(['eval_loss', 'eval_accuracy', 'eval_macro_f1', 'eval_weighted_f1', 'eval_invalid_count', 'eval_invalid_rate'])
{'eval_loss': 0.23925679922103882, 'eval_accuracy': 0.875, 'eval_macro_f1': 0.3111111111111111, 'eval_weighted_f1': 0.8166666666666667, 'eval_invalid_count': 0, 'eval_invalid_rate': 0.0}


## Hyperparameter search on validation macro-F1 only

In [ ]:
assert SELECTION_METRIC == 'macro_f1'
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
RUN_TABLES_DIR.mkdir(parents=True, exist_ok=True)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)


def trainer_tokenizer_kwargs():
    signature = inspect.signature(Seq2SeqTrainer.__init__)
    key = 'processing_class' if 'processing_class' in signature.parameters else 'tokenizer'
    return {key: tokenizer}


def make_training_arguments(config_id, config):
    training_args_kwargs = {
        'output_dir': str(CHECKPOINTS_DIR / f'config_{config_id:02d}'),
        'learning_rate': config['learning_rate'],
        'per_device_train_batch_size': config['per_device_train_batch_size'],
        'per_device_eval_batch_size': config['per_device_eval_batch_size'],
        'gradient_accumulation_steps': config['gradient_accumulation_steps'],
        'num_train_epochs': config['num_train_epochs'],
        'weight_decay': config['weight_decay'],
        'warmup_ratio': config['warmup_ratio'],
        'logging_steps': LOGGING_STEPS,
        'logging_nan_inf_filter': False,
        'save_strategy': 'epoch',
        'load_best_model_at_end': COMMON_TRAINING_CONFIG['load_best_model_at_end'],
        'metric_for_best_model': COMMON_TRAINING_CONFIG['metric_for_best_model'],
        'greater_is_better': COMMON_TRAINING_CONFIG['greater_is_better'],
        'save_total_limit': COMMON_TRAINING_CONFIG['save_total_limit'],
        'report_to': 'none',
        'remove_unused_columns': False,
        'seed': SEED,
        'data_seed': SEED,
        'fp16': COMMON_TRAINING_CONFIG['fp16'],
        'prediction_loss_only': False,
        'label_names': ['labels'],
        'predict_with_generate': COMMON_TRAINING_CONFIG['predict_with_generate'],
        'generation_max_length': MAX_TARGET_LENGTH,
        'generation_num_beams': COMMON_TRAINING_CONFIG['generation_num_beams'],
        'gradient_checkpointing': USE_GRADIENT_CHECKPOINTING,
    }
    signature = inspect.signature(Seq2SeqTrainingArguments.__init__)
    strategy_key = 'eval_strategy' if 'eval_strategy' in signature.parameters else 'evaluation_strategy'
    training_args_kwargs[strategy_key] = 'epoch'
    return Seq2SeqTrainingArguments(**training_args_kwargs)


sweep_records = []
best_run = None
for config_id, config in enumerate(FULL_FT_GRID, start=1):
    print(f'\nTraining full fine-tuning configuration {config_id}/{len(FULL_FT_GRID)}:', config)
    set_global_seed(SEED)
    run_model = new_full_ft_model().to(device)
    parameter_summary = summarize_parameters(run_model, print_summary=(config_id == 1))
    run_trainer = Seq2SeqTrainer(
        model=run_model,
        args=make_training_arguments(config_id, config),
        train_dataset=tokenized_datasets['train'],
        eval_dataset=tokenized_datasets['validation'],
        data_collator=data_collator,
        compute_metrics=compute_seq2seq_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        **trainer_tokenizer_kwargs(),
    )

    start_time = time.perf_counter()
    run_trainer.train()
    run_training_time_sec = time.perf_counter() - start_time
    validation_metrics = run_trainer.evaluate(
        tokenized_datasets['validation'],
        metric_key_prefix='validation',
    )
    assert run_trainer.state.best_model_checkpoint is not None

    record = {
        'config_id': config_id,
        **config,
        'val_macro_f1': float(validation_metrics['validation_macro_f1']),
        'val_weighted_f1': float(validation_metrics['validation_weighted_f1']),
        'val_accuracy': float(validation_metrics['validation_accuracy']),
        'val_invalid_count': int(validation_metrics['validation_invalid_count']),
        'val_invalid_rate': float(validation_metrics['validation_invalid_rate']),
        'training_time_sec': run_training_time_sec,
        'best_checkpoint': run_trainer.state.best_model_checkpoint,
        **parameter_summary,
    }
    sweep_records.append(record)
    if best_run is None or record['val_macro_f1'] > best_run['record']['val_macro_f1']:
        best_run = {
            'record': dict(record),
            'config': dict(config),
            'parameter_summary': dict(parameter_summary),
        }

    del run_trainer, run_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

sweep_results_df = (
    pd.DataFrame(sweep_records)
    .sort_values('val_macro_f1', ascending=False)
    .reset_index(drop=True)
)
sweep_results_df.to_csv(SWEEP_RESULTS_PATH, index=False)

best_config = best_run['config']
selected_training_time_sec = best_run['record']['training_time_sec']
print('\nBest full fine-tuning configuration selected only on validation macro-F1:')
print(best_config)
print('Best checkpoint:', best_run['record']['best_checkpoint'])
sweep_results_df


Training full fine-tuning configuration 1/6: {'learning_rate': 1e-05, 'num_train_epochs': 3, 'weight_decay': 0.0, 'warmup_ratio': 0.06, 'per_device_train_batch_size': 8, 'per_device_eval_batch_size': 16, 'gradient_accumulation_steps': 2}


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Total parameters:     76,961,152
Trainable parameters: 76,961,152
Trainable share:      100.0000%


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Invalid Count,Invalid Rate
1,0.719733,0.285124,0.661775,0.329273,0.551938,0,0.000000
2,0.663724,0.253654,0.707897,0.455651,0.643223,0,0.000000
3,0.646727,0.245119,0.717680,0.488748,0.663444,0,0.000000


...

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


[transformers] early stopping required metric_for_best_model, but did not find eval_macro_f1 so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1,Invalid Count,Invalid Rate
0.485465,0.192254,3,0.784766,0.694378,0.778703,0,0.000000



Best full fine-tuning configuration selected only on validation macro-F1:
{'learning_rate': 3e-05, 'num_train_epochs': 3, 'weight_decay': 0.01, 'warmup_ratio': 0.06, 'per_device_train_batch_size': 8, 'per_device_eval_batch_size': 16, 'gradient_accumulation_steps': 2}
Best checkpoint: /content/LORA-course-project/artifacts/flan_t5_small_full_ft_hparam_search/checkpoints/config_06/checkpoint-1521


,config_id,learning_rate,num_train_epochs,weight_decay,warmup_ratio,per_device_train_batch_size,per_device_eval_batch_size,gradient_accumulation_steps,val_macro_f1,val_weighted_f1,val_accuracy,val_invalid_count,val_invalid_rate,training_time_sec,best_checkpoint,total_params,trainable_params,trainable_share
0,6,0.00003,3,0.01,0.06,8,16,2,0.694378,0.778703,0.784766,0,0.0,381.808047,/content/LORA-course-project/artifacts/flan_t5...,76961152,76961152,1.0
1,5,0.00003,3,0.00,0.06,8,16,2,0.684765,0.773640,0.781272,0,0.0,374.715067,/content/LORA-course-project/artifacts/flan_t5...,76961152,76961152,1.0
2,3,0.00002,3,0.00,0.06,8,16,2,0.608372,0.733177,0.754018,0,0.0,373.951359,/content/LORA-course-project/artifacts/flan_t5...,76961152,76961152,1.0
3,4,0.00002,3,0.01,0.06,8,16,2,0.602968,0.729745,0.750524,0,0.0,379.149384,/content/LORA-course-project/artifacts/flan_t5...,76961152,76961152,1.0
4,2,0.00001,3,0.01,0.06,8,16,2,0.492252,0.665936,0.719078,0,0.0,374.742397,/content/LORA-course-project/artifacts/flan_t5...,76961152,76961152,1.0
5,1,0.00001,3,0.00,0.06,8,16,2,0.488748,0.663444,0.717680,0,0.0,383.861439,/content/LORA-course-project/artifacts/flan_t5...,76961152,76961152,1.0


## One final test evaluation

In [ ]:
assert FINAL_TEST_EVALUATION_ONLY is True
assert best_run is not None

set_global_seed(SEED)
best_model = AutoModelForSeq2SeqLM.from_pretrained(
    best_run['record']['best_checkpoint']
).to(device)
best_model.eval()

final_eval_args = Seq2SeqTrainingArguments(
    output_dir=str(ARTIFACTS_DIR / 'final_evaluation'),
    per_device_eval_batch_size=best_config['per_device_eval_batch_size'],
    report_to='none',
    remove_unused_columns=False,
    fp16=COMMON_TRAINING_CONFIG['fp16'],
    prediction_loss_only=False,
    label_names=['labels'],
    predict_with_generate=COMMON_TRAINING_CONFIG['predict_with_generate'],
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=COMMON_TRAINING_CONFIG['generation_num_beams'],
)
best_trainer = Seq2SeqTrainer(
    model=best_model,
    args=final_eval_args,
    data_collator=data_collator,
    compute_metrics=compute_seq2seq_metrics,
    **trainer_tokenizer_kwargs(),
)
test_prediction_output = best_trainer.predict(
    tokenized_datasets['test'],
    metric_key_prefix='test',
)

test_generated_texts = decode_token_ids(test_prediction_output.predictions)
test_predictions = np.array([
    normalize_generated_label(text) for text in test_generated_texts
])
test_labels = raw_dataframes['test']['label'].to_numpy()
test_metrics = main_metrics(test_labels, test_predictions)

test_invalid_mask = test_predictions == INVALID_LABEL_ID
test_invalid_count = int(np.sum(test_invalid_mask))
test_invalid_rate = float(test_invalid_count / len(test_predictions)) if len(test_predictions) else 0.0

test_classification_report = classification_report(
    test_labels,
    test_predictions,
    labels=LABEL_IDS,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
test_classification_report_df = (
    pd.DataFrame(test_classification_report).transpose().rename_axis('class').reset_index()
)
test_confusion_matrix_df = pd.DataFrame(
    confusion_matrix(test_labels, test_predictions, labels=LABEL_IDS),
    index=[f'true_{class_name}' for class_name in CLASS_NAMES],
    columns=[f'pred_{class_name}' for class_name in CLASS_NAMES],
)

test_predictions_df = raw_dataframes['test'].copy()
test_predictions_df['true_label'] = test_labels
test_predictions_df['true_label_name'] = test_predictions_df['true_label'].map(id2label)
test_predictions_df['generated_text'] = test_generated_texts
test_predictions_df['pred_label'] = test_predictions
test_predictions_df['pred_label_name'] = test_predictions_df['pred_label'].map(
    lambda class_id: id2label.get(int(class_id), 'INVALID')
)
test_predictions_df['prediction_valid'] = ~test_invalid_mask
test_predictions_df['correct'] = test_predictions_df['true_label'] == test_predictions_df['pred_label']

generation_diagnostics_df = pd.DataFrame([{
    'split': 'test',
    'invalid_prediction_count': test_invalid_count,
    'invalid_prediction_rate': test_invalid_rate,
}])

print('Final test metrics from the validation-selected full fine-tuning checkpoint:')
print(test_metrics)
print('Invalid generated labels:', test_invalid_count, f'({test_invalid_rate:.4%})')

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Final test metrics from the validation-selected full fine-tuning checkpoint:
{'accuracy': 0.8082077051926299, 'macro_f1': 0.7272880720289759, 'weighted_f1': 0.8064163566472557}
Invalid generated labels: 0 (0.0000%)


## Save results and best model checkpoint

In [ ]:
RUN_TABLES_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_RESULT_COLUMNS = [
    'model',
    'adaptation',
    'test_macro_f1',
    'test_weighted_f1',
    'test_accuracy',
    'bearish_f1',
    'bullish_f1',
    'neutral_f1',
    'total_params',
    'trainable_params',
    'trainable_share',
    'training_time_sec',
]

selected_parameter_summary = best_run['parameter_summary']
final_test_row = {
    'model': MODEL_ID,
    'adaptation': 'full fine-tuning',
    'test_macro_f1': test_metrics['macro_f1'],
    'test_weighted_f1': test_metrics['weighted_f1'],
    'test_accuracy': test_metrics['accuracy'],
    'bearish_f1': float(test_classification_report['Bearish']['f1-score']),
    'bullish_f1': float(test_classification_report['Bullish']['f1-score']),
    'neutral_f1': float(test_classification_report['Neutral']['f1-score']),
    'total_params': selected_parameter_summary['total_params'],
    'trainable_params': selected_parameter_summary['trainable_params'],
    'trainable_share': selected_parameter_summary['trainable_share'],
    'training_time_sec': selected_training_time_sec,
}
final_test_results_df = pd.DataFrame([final_test_row], columns=FINAL_RESULT_COLUMNS)
assert list(final_test_results_df.columns) == FINAL_RESULT_COLUMNS
final_test_results_df.to_csv(FINAL_TEST_RESULTS_PATH, index=False)

test_predictions_df.to_csv(
    RUN_TABLES_DIR / 'flan_t5_small_full_ft_best_hparam_test_predictions.csv', index=False
)
test_classification_report_df.to_csv(
    RUN_TABLES_DIR / 'flan_t5_small_full_ft_best_hparam_test_classification_report.csv', index=False
)
test_confusion_matrix_df.to_csv(
    RUN_TABLES_DIR / 'flan_t5_small_full_ft_best_hparam_test_confusion_matrix.csv'
)
generation_diagnostics_df.to_csv(GENERATION_DIAGNOSTICS_PATH, index=False)

best_model.save_pretrained(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
experiment_config = {
    'dataset_name': DATASET_NAME,
    'model': MODEL_ID,
    'adaptation': 'full fine-tuning',
    'seed': SEED,
    'max_source_length': MAX_SOURCE_LENGTH,
    'max_target_length': MAX_TARGET_LENGTH,
    'prompt_template': PROMPT_TEMPLATE,
    'selection_metric': SELECTION_METRIC,
    'selection_split': 'validation',
    'test_evaluation_after_validation_selection_only': FINAL_TEST_EVALUATION_ONLY,
    'fixed_grid': FULL_FT_GRID,
    'fixed_training_settings': {
        'generation_max_length': MAX_TARGET_LENGTH,
        'generation_num_beams': GENERATION_NUM_BEAMS,
        'fp16': COMMON_TRAINING_CONFIG['fp16'],
        'use_gradient_checkpointing': USE_GRADIENT_CHECKPOINTING,
    },
    'selected_config': best_config,
    'selected_best_checkpoint': best_run['record']['best_checkpoint'],
    'selected_training_time_sec': selected_training_time_sec,
    'sweep_results_csv': str(SWEEP_RESULTS_PATH),
    'final_test_results_csv': str(FINAL_TEST_RESULTS_PATH),
    'generation_diagnostics_csv': str(GENERATION_DIAGNOSTICS_PATH),
}
with EXPERIMENT_CONFIG_PATH.open('w', encoding='utf-8') as config_file:
    json.dump(experiment_config, config_file, indent=2)

print('Saved validation sweep table to:', SWEEP_RESULTS_PATH)
print('Saved final best-model test table to:', FINAL_TEST_RESULTS_PATH)
print('Saved generation diagnostics to:', GENERATION_DIAGNOSTICS_PATH)
print('Saved validation-selected full fine-tuning model and tokenizer to:', BEST_MODEL_DIR)
final_test_results_df

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved validation sweep table to: /content/LORA-course-project/reports/tables/flan_t5_small_full_ft/flan_t5_small_full_ft_hparam_search_validation_results.csv
Saved final best-model test table to: /content/LORA-course-project/reports/tables/flan_t5_small_full_ft/flan_t5_small_full_ft_best_hparam_test_results.csv
Saved generation diagnostics to: /content/LORA-course-project/reports/tables/flan_t5_small_full_ft/flan_t5_small_full_ft_best_hparam_generation_diagnostics.csv
Saved validation-selected full fine-tuning model and tokenizer to: /content/LORA-course-project/artifacts/flan_t5_small_full_ft_hparam_search/best_model


,model,adaptation,test_macro_f1,test_weighted_f1,test_accuracy,bearish_f1,bullish_f1,neutral_f1,total_params,trainable_params,trainable_share,training_time_sec
0,google/flan-t5-small,full fine-tuning,0.727288,0.806416,0.808208,0.605697,0.689947,0.88622,76961152,76961152,1.0,381.808047


## Save artifacts to Google Drive and download locally

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

if 'DRIVE_ARTIFACTS_ROOT' not in globals():
    raise RuntimeError(
        'DRIVE_ARTIFACTS_ROOT is not defined. Run the Google Drive setup cell at the beginning before training.'
    )

RUN_ARTIFACT_SOURCES = {
    'artifacts': ARTIFACTS_DIR,
    'tables': RUN_TABLES_DIR,
}
missing_sources = [str(source_path) for source_path in RUN_ARTIFACT_SOURCES.values() if not source_path.exists()]
if missing_sources:
    raise FileNotFoundError(f'Expected artifact source directories were not found: {missing_sources}')

bundle_root = Path('/content') / f'{RUN_NAME}_artifact_bundle'
local_zip_base = Path('/content') / f'{RUN_NAME}_artifacts'
local_zip_path = local_zip_base.with_suffix('.zip')

if bundle_root.exists():
    shutil.rmtree(bundle_root)
if local_zip_path.exists():
    local_zip_path.unlink()

bundle_root.mkdir(parents=True, exist_ok=True)
for bundle_name, source_path in RUN_ARTIFACT_SOURCES.items():
    shutil.copytree(source_path, bundle_root / bundle_name, dirs_exist_ok=True)

local_zip_path = Path(shutil.make_archive(str(local_zip_base), 'zip', root_dir=bundle_root))
DRIVE_ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
drive_zip_path = DRIVE_ARTIFACTS_ROOT / local_zip_path.name
shutil.copy2(local_zip_path, drive_zip_path)

print(f'Saved artifact ZIP to Google Drive: {drive_zip_path}')
print(f'ZIP size: {drive_zip_path.stat().st_size / (1024 ** 2):.2f} MB')

try:
    files.download(str(drive_zip_path))
    print('Started browser download for artifact ZIP.')
except Exception as download_error:
    print('Browser download did not start, but the Google Drive backup is complete.')
    print(f'Download error: {download_error}')

Saved artifact ZIP to Google Drive: /content/drive/MyDrive/lora_course_project/artifacts/flan_t5_small_full_ft_hparam_search_artifacts.zip
ZIP size: 4759.78 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Started browser download for artifact ZIP.
